In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import MNIST

In [3]:
from torchvision.transforms import transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.6) , (0.6))
])
trainset = MNIST(root = "./data" , train = True , transform = transform , download = True)
testset = MNIST(root = "./data" , train = False , transform = transform , download = True)

In [4]:
from torch.utils.data import DataLoader

trainLoader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testLoader = DataLoader(testset , batch_size = 64 , shuffle = False)

In [5]:
img , label = trainset[0]
print(img.shape)
print(label)

torch.Size([1, 28, 28])
5


In [6]:
import torch.optim as optim

## Build the CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN , self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(1 , 32 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(32 , 64 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(64 , 128 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

        )

        self.fc_layer = nn.Sequential(
            nn.Linear(3 * 3 *128 , 256),
            nn.ReLU(),
            nn.Linear(256 , 10)
        )


    def forward(self , x):
        x = self.conv_layer(x)
        x = x.view(x.size(0) , -1)
        x = self.fc_layer(x)

        return x      
        

In [8]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [82]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images , labels in trainLoader:
        optimizer.zero_grad()

        outputs = model.forward(images)
        loss = criterion(outputs , labels)
        loss.backward()
        optimizer.step()  #parameter update

        running_loss += loss.item()

    print(f"{epoch + 1} / {epochs} and loss = {running_loss/len(trainLoader)}")

1 / 10 and loss = 0.16468951744171403
2 / 10 and loss = 0.04232347898407659
3 / 10 and loss = 0.029463121882726827
4 / 10 and loss = 0.02371873366626766
5 / 10 and loss = 0.016956866176583672
6 / 10 and loss = 0.015338477969940706
7 / 10 and loss = 0.012675397910618431
8 / 10 and loss = 0.01081265469041719
9 / 10 and loss = 0.009813968703957104
10 / 10 and loss = 0.007759858051960923


In [83]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images , labels in testLoader:
        outputs = model.forward(images)
        _  , predicted = torch.max(outputs , 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


    print(f"accuracy : {correct_labels /total_labels *100 }")

    

accuracy : 99.29


## RNN Implementation 

In [21]:
class RNN(nn.Module):
    def __init__(self , in_size = 1 , hidden_size = 64 , num_layers = 2):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #Rnn layer
        self.rnn = nn.RNN(in_size , hidden_size , num_layers, batch_first = True)

        #fc layer
        self.fc_layer = nn.Linear(hidden_size , 10)

    def forward(self , x):
        out , _ = self.rnn(x)
        out = self.fc_layer(out[: , -1 , :])

        return out

In [22]:
model_rnn = RNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_rnn.parameters())

In [24]:
epochs = 10

for epoch in range(epochs):
    model_rnn.train()

    for images , labels in trainLoader:
    
        optimizer.zero_grad()

        outputs = model_rnn(images.squeeze())
        loss = criterion(outputs ,labels)
        loss.backward()
        optimizer.step()

    print(f"{epoch + 1}/{epochs} and loss = {loss.item()}")

1/10 and loss = 0.5752966403961182
2/10 and loss = 0.47924724221229553
3/10 and loss = 0.09073353558778763
4/10 and loss = 0.04376450926065445
5/10 and loss = 0.42490437626838684
6/10 and loss = 0.03476357460021973
7/10 and loss = 0.18420077860355377
8/10 and loss = 0.31728050112724304
9/10 and loss = 0.09372732788324356
10/10 and loss = 0.074031300842762


In [25]:
correct_values = 0.0
total_values = 0.0


model.eval()
with torch.no_grad():
    for images , labels in testLoader:

        outputs = model_rnn(images.squeeze())
        _  , predicted = torch.max(outputs , 1)

        correct_values += (predicted == labels).sum().item()
        total_values += labels.size(0)

    print(f"accuracy : {correct_values /total_values *100 }")

accuracy : 96.63000000000001
